# AkStudioFreeai — Personal Speaking Voice Clone

Create speech in a voice you own or have explicit permission to use.

This notebook uses OpenVoice V2 for tone-color conversion and MeloTTS for base speech.
Supported MeloTTS languages include English (including Indian English), Spanish, French,
Chinese, Japanese and Korean. Hindi is not listed as a supported MeloTTS language here.

Free Colab GPU availability is not guaranteed. Keep downloaded outputs safe.


In [ ]:
# STEP 1 — Check runtime
import sys, subprocess
print("Python:", sys.version)
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "No NVIDIA GPU detected.")


In [ ]:
# STEP 2 — Install OpenVoice V2 and MeloTTS
%cd /content
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!git clone --depth 1 https://github.com/myshell-ai/OpenVoice.git
%cd /content/OpenVoice
!pip install -q -e .
!pip install -q git+https://github.com/myshell-ai/MeloTTS.git
!python -m unidic download


In [ ]:
# STEP 3 — Download official OpenVoice V2 checkpoints
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="myshell-ai/OpenVoiceV2",
    local_dir="/content/OpenVoice/checkpoints_v2"
)
print("Checkpoint download complete.")


## STEP 4 — Upload your own reference voice

Use a clean recording with one speaker, minimal background noise and no music.
A longer, clear sample generally gives better voice identity extraction.
Only upload a voice you own or have permission to use.


In [ ]:
# STEP 4 — Upload reference audio
from google.colab import files
from pathlib import Path
import shutil, os

uploaded = files.upload()
audio_names = [n for n in uploaded if Path(n).suffix.lower() in
               {".wav", ".mp3", ".m4a", ".webm", ".ogg", ".flac"}]
if not audio_names:
    raise RuntimeError("Upload a supported audio recording.")
reference_speaker = "/content/reference_voice.wav"
!ffmpeg -y -i "{audio_names[0]}" -vn -ac 1 -ar 24000 "$reference_speaker"
print("Reference voice ready:", reference_speaker)


In [ ]:
# STEP 5 — Load OpenVoice V2
import torch
from openvoice import se_extractor
from openvoice.api import ToneColorConverter

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ckpt_converter = "/content/OpenVoice/checkpoints_v2/converter"
tone_color_converter = ToneColorConverter(
    f"{ckpt_converter}/config.json", device=device
)
tone_color_converter.load_ckpt(f"{ckpt_converter}/checkpoint.pth")
target_se, audio_name = se_extractor.get_se(
    reference_speaker, tone_color_converter, vad=True
)
print("Reference voice embedding extracted.")


## STEP 6 — Enter text and choose a supported language

Use EN for English. MeloTTS also documents ES, FR, ZH, JP and KR.
Hindi is not a documented language option in this demo.


In [ ]:
# STEP 6 — Generate speech in your reference voice
from melo.api import TTS
from IPython.display import Audio, display

text_to_speak = "Hello! This is a sample of my personal AI voice."
language = "EN"  # EN, ES, FR, ZH, JP, KR
speed = 1.0

if not text_to_speak.strip():
    raise ValueError("Enter text before generating.")

tts = TTS(language=language, device=device)
speaker_ids = tts.hps.data.spk2id
print("Available base speakers:", list(speaker_ids.keys()))

# Prefer Indian English speaker when available; otherwise use first available speaker.
speaker_key = next(
    (k for k in speaker_ids if "india" in k.lower()),
    next(iter(speaker_ids))
)
speaker_id = speaker_ids[speaker_key]
base_wav = "/content/base_speech.wav"
output_wav = "/content/AkStudioFreeai_Speaking_Clone.wav"

tts.tts_to_file(text_to_speak, speaker_id, base_wav, speed=speed)
source_se = torch.load(
    f"/content/OpenVoice/checkpoints_v2/base_speakers/ses/{speaker_key.lower().replace('_','-')}.pth",
    map_location=device,
    weights_only=False
)
tone_color_converter.convert(
    audio_src_path=base_wav,
    src_se=source_se,
    tgt_se=target_se,
    output_path=output_wav,
    message="@AkStudioFreeai"
)
print("Generated:", output_wav)
display(Audio(output_wav))


In [ ]:
# STEP 7 — Download generated voice
from google.colab import files
files.download("/content/AkStudioFreeai_Speaking_Clone.wav")


## Troubleshooting

- If no GPU is available, try CPU; generation may be slow.
- If a language or speaker checkpoint is missing, check the official OpenVoice and MeloTTS documentation.
- Hindi is not guaranteed by this notebook.
- Save your WAV output before the Colab runtime ends.

Official projects:
- https://github.com/myshell-ai/OpenVoice
- https://github.com/myshell-ai/MeloTTS
